# El vaso no basta · contexto largo con la misma GRU

**Meta.** Misma máquina del lab 0, distinto $L\in\{32,64,128,256\}$. Mismas épocas, mismo LR, mismo ancho. Solo cambia cuántas letras ve de un jalón.

## Dummy

La hipótesis ingenua: “si le doy más letras atrás, predice mejor”. Suena lógico. En una GRU es **falso a menudo**.

Por qué: la GRU no recibe esas letras sueltas. Recibe un resumen $h$ (el vaso). Alargar $L$ de 64 a 256 le da **más tarea al vaso, no más vaso**. Además el gradiente tiene que viajar 256 pasos hacia atrás; a menudo se diluye y el modelo ni se entera de la letra lejana.

Vas a ver curvas que *no* bajan al subir $L$. Esa gráfica es el argumento para cambiar de máquina: **atención**. Aquí se acaba la GRU como eje del curso. Una sesión, no seis.

## Matemática

La información que el prefijo lejano puede meter en la predicción está acotada por la capacidad del vaso:

$$
I(x_{\le t-L};\,x_t\mid h_{t-1})\;\le\;I(h_{t-1};\,x_t),
$$

y $h_{t-1}\in\mathbb{R}^{d}$ tiene dimensión **fija**. Alargar $L$ no aumenta $d$. Si el gradiente $\partial\mathcal{L}/\partial h_{t-L}$ se desvanece, el vaso ni siquiera usa lo que sí cabría. Por eso “más contexto” no es monótono.

En cristiano: $I$ es “cuánto me entero de $A$ si veo $B$”. El pasado lejano solo puede influir *a través* del post-it $h$. Un post-it de 128 números no se vuelve más grande porque leas 256 letras.

## Cómo leer la tabla

Un experimento justo cambia **una sola cosa**. Si entrenas 4 000 ventanas de cada largo, el modelo con $L=256$ lee 8 veces más letras que el de $L=32$ y gana por eso, no por el contexto. Si igualas las letras pero no los pasos, el de $L=256$ da 8 veces menos pasos de gradiente y pierde por eso. Aquí igualamos las dos cosas: mismo total de letras y mismo número de pasos (batch de $2048/L$ ventanas).

Y medimos dos cosas en la **misma** validación (ventanas de 256 letras, para que todos los modelos puedan leer el mismo pasado):

1. La val loss total por $L$.
2. La pérdida **por posición** dentro de la ventana: ¿predice mejor la letra 200 que la letra 20? Si el pasado lejano ayudara, la pérdida seguiría bajando con la posición.


In [1]:
# Colab: descomenta las dos líneas de abajo (una vez).
# %pip install -q torch
# !wget -q https://raw.githubusercontent.com/NoSoyRo/diplomado-ia-llms/main/datos/quijote.txt

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)  # misma corrida que la guardada (en CPU)
from pathlib import Path

# Local: ../datos/quijote.txt (o ../../ desde extra/). Colab: el wget lo deja junto al notebook.
TEXT_PATH = next(
    (str(p) for p in (Path("quijote.txt"), Path("../datos/quijote.txt"), Path("../../datos/quijote.txt")) if p.exists()),
    "quijote.txt",
)
print(f"device={DEVICE}  archivo={TEXT_PATH}")


device=cpu  archivo=../datos/quijote.txt


In [2]:
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().replace("\r\n", "\n").replace("\r", "\n")


def build_vocab(text: str):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos


def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)


def decode(ids: List[int], itos: dict) -> str:
    return "".join(itos[i] for i in ids)


In [3]:
class CharWindowDataset(Dataset):
    """Ventanas x → y corrido +1. Si max_windows no es None, muestrea inicios al azar.

    Dummy: el Quijote tiene ~2M letras. Recorrer TODAS las ventanas en CPU no cabe
    en una sesión. Muestrear M inicios es el mismo examen, con menos filas.
    """

    def __init__(self, data_ids: torch.Tensor, seq_len: int, max_windows: Optional[int] = 8000, seed: int = 0):
        self.data = data_ids
        self.seq_len = seq_len
        n = max(0, len(self.data) - self.seq_len - 1)
        if max_windows is not None and n > max_windows:
            g = torch.Generator().manual_seed(seed)
            self.index = torch.randint(0, n, (max_windows,), generator=g)
        else:
            self.index = torch.arange(n)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i: int):
        idx = int(self.index[i])
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


In [4]:
class CharGRULM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 128, hidden_dim: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T) → logits (B, T, V) y vaso final hN
        emb = self.embed(x)
        out, hN = self.gru(emb, h0)
        logits = self.fc(out)
        return logits, hN


In [5]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        B, T, V = logits.shape
        loss = criterion(logits.view(B * T, V), y.view(B * T))
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


In [6]:
@torch.no_grad()
def generate(model, prompt: str, stoi: dict, itos: dict, max_new_tokens: int = 400, temperature: float = 1.0, device="cpu"):
    """Auto-regresivo: predice, pega, predice otra vez.

    El primer token nuevo sale de los logits del ÚLTIMO carácter del prompt.
    (Antes el código volvía a comer ese carácter: el texto salía corrido.)
    """
    model.eval()
    ids = [stoi[ch] for ch in prompt if ch in stoi] or [0]
    out_ids = ids.copy()
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    logits, h = model(x, h0=None)
    for _ in range(max_new_tokens):
        z = logits[0, -1, :] / max(1e-8, temperature)
        probs = torch.softmax(z, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        out_ids.append(next_id)
        inp = torch.tensor([[next_id]], dtype=torch.long, device=device)
        logits, h = model(inp, h0=h)
    return decode(out_ids, itos)


## Experimento

Cuatro $L$, mismo presupuesto (256 000 letras, 3 épocas, 125 pasos por época), misma semilla. Tarda ~20 s en CPU.

Entregable en una frase: ¿el vaso basta para un noticiero de 10 notas? (spoiler de producto: no. Las notas 1 y 10 no caben en $h$.)

In [7]:
@dataclass
class ExpConfig:
    text_path: str = TEXT_PATH
    device: str = DEVICE
    emb_dim: int = 128
    hidden_dim: int = 128
    num_layers: int = 1
    lr: float = 3e-4
    epochs: int = 3
    letras_por_epoca: int = 256_000   # mismo total para todos los L
    letras_por_batch: int = 2_048     # mismo número de pasos para todos los L
    val_len: int = 256                # validación común: todos leen el mismo pasado
    val_windows: int = 600
    seq_lens: Tuple[int, ...] = (32, 64, 128, 256)
    tramos: Tuple[Tuple[int, int], ...] = ((1, 8), (9, 32), (33, 64), (65, 128), (129, 256))
    prompt: str = "En un lugar de la Mancha "
    temperature: float = 1.0
    gen_tokens: int = 250


@torch.no_grad()
def perdida_por_posicion(model, loader, device) -> torch.Tensor:
    """Promedio de -log P(y_t) para cada posición t de la ventana."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="none")
    total, n = None, 0
    for x, y in loader:
        logits, _ = model(x.to(device))
        por_t = criterion(logits.transpose(1, 2), y.to(device)).sum(0)
        total = por_t if total is None else total + por_t
        n += len(x)
    return total.cpu() / n


def run_experiment(cfg: ExpConfig):
    text = load_text(cfg.text_path)
    stoi, itos = build_vocab(text)
    data_ids = encode(text, stoi)
    split = int(0.9 * len(data_ids))
    train_ids, val_ids = data_ids[:split], data_ids[split:]
    device = torch.device(cfg.device)
    val_ds = CharWindowDataset(val_ids, cfg.val_len, max_windows=cfg.val_windows, seed=1)
    val_loader = DataLoader(val_ds, batch_size=50, shuffle=False)
    results = []

    print(f"Vocab={len(stoi)} | chars={len(data_ids)} | device={cfg.device}")
    for L in cfg.seq_lens:
        torch.manual_seed(0)
        train_ds = CharWindowDataset(train_ids, L, max_windows=cfg.letras_por_epoca // L, seed=0)
        train_loader = DataLoader(train_ds, batch_size=cfg.letras_por_batch // L, shuffle=True, drop_last=True)
        model = CharGRULM(len(stoi), cfg.emb_dim, cfg.hidden_dim, cfg.num_layers).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        print(f"\nL={L:3d} | ventanas={len(train_ds):5d} | batch={cfg.letras_por_batch // L:2d} | pasos/época={len(train_loader)}")
        for epoch in range(1, cfg.epochs + 1):
            tr = train_one_epoch(model, train_loader, optimizer, device)
            print(f"  Epoch {epoch:02d} | train_loss={tr:.4f}")

        pos = perdida_por_posicion(model, val_loader, device)
        sample = generate(model, cfg.prompt, stoi, itos, cfg.gen_tokens, cfg.temperature, str(device))
        results.append((L, pos, sample))
    return results


cfg = ExpConfig()
results = run_experiment(cfg)

print("\n" + "=" * 86)
print("Validación común (ventanas de 256). Columnas: pérdida media en esas posiciones de la ventana.")
print(f"{'L':>4} {'val':>7} | " + " ".join(f"{f'{a}-{b}':>8}" for a, b in cfg.tramos))
for L, pos, _ in results:
    print(f"{L:>4} {pos.mean():>7.3f} | " + " ".join(f"{pos[a - 1:b].mean():>8.3f}" for a, b in cfg.tramos))

print("\n" + "=" * 86)
print("MUESTRAS (una por L, misma temperatura)")
for L, _, sample in results:
    print("\n" + "-" * 70)
    print(f"L={L}")
    print(sample[:600])


Vocab=91 | chars=2097953 | device=cpu



L= 32 | ventanas= 8000 | batch=64 | pasos/época=125


  Epoch 01 | train_loss=3.1934


  Epoch 02 | train_loss=2.4274


  Epoch 03 | train_loss=2.2431

L= 64 | ventanas= 4000 | batch=32 | pasos/época=125


...

  Epoch 02 | train_loss=2.4210


  Epoch 03 | train_loss=2.2382

L=256 | ventanas= 1000 | batch= 8 | pasos/época=125


  Epoch 01 | train_loss=3.1689


  Epoch 02 | train_loss=2.4178


  Epoch 03 | train_loss=2.2343

Validación común (ventanas de 256). Columnas: pérdida media en esas posiciones de la ventana.
   L     val |      1-8     9-32    33-64   65-128  129-256
  32   2.188 |    2.283    2.190    2.187    2.180    2.185
  64   2.187 |    2.311    2.186    2.186    2.178    2.184
 128   2.190 |    2.338    2.191    2.188    2.181    2.186
 256   2.195 |    2.368    2.194    2.192    2.185    2.190

MUESTRAS (una por L, misma temperatura)

----------------------------------------------------------------------
L=32
En un lugar de la Mancha dijas su aleradies
otral se vali laca. Qería on en ensy la moñor diéca, es punde noro ñe
sicial y, la én de qundestro ere ylo ciampa.

-T
y core so ta bijorvano yncualjpúo estrol bapestodio..
edirvigrpor, dejer er der píiveson
quos ñíre Quintedo vede

----------------------------------------------------------------------
L=64
En un lugar de la Mancha dijas su alesadies
otral sertasi laca. Qería on en ensy la moñor diéca, es pun

## Lo que salió (corrida guardada)

- **Más contexto no ayudó.** Con el mismo presupuesto, la val loss queda en ~2.19 para los cuatro $L$; $L=256$ es incluso un pelo peor.
- **La curva por posición se aplana muy pronto.** De la letra 1–8 a la 9–32 la pérdida baja (el modelo arma la palabra en curso). De ahí en adelante es plana: predecir la letra 200 cuesta lo mismo que la letra 40. Lo que pasó hace más de ~30 letras no llega a la predicción.
- **Las muestras salen casi idénticas** (misma semilla): los cuatro modelos aprendieron lo mismo, sin importar cuánto contexto les diste.
- **Eso es el vaso.** $h$ tiene 128 números. Darle 256 letras no lo hace más grande, y el gradiente que viaja 256 pasos hacia atrás llega casi en cero.

Si quitas la igualación (pon `letras_por_epoca = 4000 * L`), $L=256$ "gana": lee 8 veces más texto. Esa es la trampa que más se repite en experimentos de contexto.

## Entregable

1. La tabla de arriba y una frase: ¿a partir de qué posición deja de bajar la pérdida?
2. La misma tabla con `hidden_dim=256`. ¿Se mueve el punto donde se aplana? (Más vaso no es más memoria lejana si el gradiente no llega.)
3. Una frase de producto: ¿por qué un noticiero de 10 notas necesita otra máquina?

## Para romperlo

- `letras_por_epoca = 4000 * L` (el experimento ingenuo). ¿Qué $L$ gana y por qué es trampa?
- `num_layers=2`. ¿Cambia la curva por posición?
- En el notebook 03, mide `perdida_por_posicion` con el mini-Transformer (adapta el `model(x)` porque no devuelve `h`). ¿Se aplana igual de pronto?